In [9]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np

#### Bootstrapping Data Analysis:

In [10]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np

eval_result_df_path = '/Users/nicolemeister/Desktop/STANFORD/benchmarking-distributional-alignment/results/distributional_alignment_task_all_groups.csv'
df=pd.read_csv(eval_result_df_path)

# drop all the rows that have Pew_American_Trends_Panel_disagreement_500
df = df[df['Wave'] != 'Pew_American_Trends_Panel_disagreement_500']

values_to_keep = ['POLPARTY', 'SEX', 'RACE']

df = df[df['Demographic Group/Avg'].isin(values_to_keep)]

values_to_keep = ['Democrat', 'Republican', 'Male', 'Female', 'Black', 'White']

df = df[df['Demographic/Avg'].isin(values_to_keep)]
# df = df.drop_duplicates(subset=['Task Type', 'Model', 'Dataset', 'Wave', 'Demographic Group/Avg', 'Demographic/Avg', 'Output Type',])


In [11]:
def compute_both(opinionqa_data, nytimes_data, human_data=False, bootstrap=True):     
    opinionqa, nytimes = [], []

    # convert the list of lists into a full list 
    if not human_data: 
        for lst in opinionqa_data:
            opinionqa.extend(eval(lst))
            # except: opinionqa.extend(list(lst))
    else: opinionqa = opinionqa_data

    # convert the list of lists into a full list 
    if not human_data: 
        for lst in nytimes_data:
            nytimes.extend(eval(lst))
    else: nytimes = nytimes_data


    num_bootstraps = 1000
    def compute_statistic(opinionqa, nytimes):
        
        # Calculate averages
        try: average_opinionqa= np.mean(opinionqa)
        except: print(opinionqa)
        try: average_nytimes = np.mean(nytimes)
        except: print(nytimes)
        weighted_average = 0.5 * average_opinionqa + 0.5 * average_nytimes
        return weighted_average
    # Bootstrapping process
    bootstrap_statistics = []
    for _ in range(num_bootstraps):
        bootstrap_sample_oqa = np.random.choice(opinionqa, size=len(opinionqa), replace=True)
        bootstrap_sample_nyt = np.random.choice(nytimes, size=len(nytimes), replace=True)
        statistic = compute_statistic(bootstrap_sample_oqa, bootstrap_sample_nyt)
        bootstrap_statistics.append(statistic)

    # 95% confidence interval
    confidence_level = 0.95
    alpha = (1 - confidence_level) / 2
    lower_percentile = alpha * 100
    upper_percentile = (1 - alpha) * 100
    lower_bound = np.percentile(bootstrap_statistics, lower_percentile)
    upper_bound = np.percentile(bootstrap_statistics, upper_percentile)
    return np.mean(bootstrap_statistics), (upper_bound-lower_bound)/2


In [12]:
def compute_one(input_data):     
    data = []

    # convert the list of lists into a full list 
    for lst in input_data:
        try: data.extend(eval(lst))
        except: data.append(lst)

    num_bootstraps = 1000

    def compute_statistic(sample):
        # todo: weighted average so that nytimes is weighted 50% and opinionqa is weighted 50%
        return np.mean(sample)  

    # Bootstrapping process
    bootstrap_statistics = []
    for _ in range(num_bootstraps):
        bootstrap_sample = np.random.choice(data, size=len(data), replace=True)
        statistic = compute_statistic(bootstrap_sample)
        bootstrap_statistics.append(statistic)

    # 95% confidence interval
    confidence_level = 0.95
    alpha = (1 - confidence_level) / 2
    lower_percentile = alpha * 100
    upper_percentile = (1 - alpha) * 100
    lower_bound = np.percentile(bootstrap_statistics, lower_percentile)
    upper_bound = np.percentile(bootstrap_statistics, upper_percentile)
    
    return np.mean(bootstrap_statistics), (upper_bound-lower_bound)/2

In [13]:
def subtract_lists(list1, list2):
    length=min(eval(list1), eval(list2))
    return [list(np.array(eval(a))[:length]- np.array(eval(b))[:length]) for a, b in zip(list1, list2)]



In [14]:
import os
final_results_path='/Users/nicolemeister/Desktop/STANFORD/benchmarking-distributional-alignment/results/distributional_alignment_task_final_leaderboard.csv'
if os.path.exists(final_results_path):
    final_results_df = pd.read_csv(final_results_path)
else: final_results_df = pd.DataFrame(columns=['Model Name', 'Dataset', 'Task Type', 'Alignment Mean', 'Alignment Error'])

In [15]:
final_results_df = pd.DataFrame(columns=['Model Name', 'Dataset', 'Task Type', 'Alignment Mean', 'Alignment Error'])

output_type_to_latex = {'express_distribution': "V", 'sequence': "Seq", 'model_logprobs': "Log-p", 'rescaled_model_logprobs': "TS-Log-p"}

models = ['gpt-3.5-turbo-0125', 'gpt-4', 'anthropic_haiku', 'anthropic_opus', 'llama3-70b']
output_types = ['express_distribution', 'sequence', 'model_logprobs', 'rescaled_model_logprobs']
# output_types = ['sequence']
# task_types = ['task0', 'task1', 'task3_easy_hard', 'task3_easy', 'task3_all_but_one']
task_types = ['task0', 'task1', 'task3_easy_hard']
# output_types = ['express_distribution', 'sequence']
for dataset in ['opinionqa', 'nytimes']:
  for task_type in task_types:
    print(task_type)
    for model in models: 
      print(model)
      for output_type in output_types: 
        print(output_type)
        
        input_data = df[(df.Model==model) & (df['Output Type']==output_type) & (df['Dataset']==dataset) & (df['Task Type']==task_type)]['all_tvs']
        mean, bs = compute_one(input_data)

        # Adding a data entry
        new_entry = pd.DataFrame({
            'Model Name': ["{} ({})".format(model, output_type_to_latex[output_type])],
            'Dataset': [dataset],
            'Task Type': [task_type],
            'Alignment Mean': [mean],
            'Alignment Error': [bs]
        })

        # Adding the new entry to the dataframe
        final_results_df = pd.concat([final_results_df, new_entry], ignore_index=True)

        print('{:.4f} +/- {:.4f}'.format(mean, bs))


task0
gpt-3.5-turbo-0125
express_distribution
0.3931 +/- 0.0135
sequence
0.3976 +/- 0.0139
model_logprobs
0.5399 +/- 0.0168
rescaled_model_logprobs


/var/folders/_r/w12mdwc56wb9k_g0q82pk8400000gn/T/ipykernel_73487/3353673578.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_results_df = pd.concat([final_results_df, new_entry], ignore_index=True)


0.3982 +/- 0.0124
gpt-4
express_distribution
0.4001 +/- 0.0147
sequence
0.3891 +/- 0.0151
model_logprobs
0.7133 +/- 0.0185
rescaled_model_logprobs
0.5025 +/- 0.0161
anthropic_haiku
express_distribution
0.4278 +/- 0.0163
sequence
0.3719 +/- 0.0136
model_logprobs
nan +/- nan
rescaled_model_logprobs
nan +/- nan
anthropic_opus
express_distribution


/Users/nicolemeister/.virtualenvs/tatsu/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/nicolemeister/.virtualenvs/tatsu/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


0.4514 +/- 0.0137
sequence
0.4845 +/- 0.0198
model_logprobs
nan +/- nan
rescaled_model_logprobs
nan +/- nan
llama3-70b
express_distribution
0.3965 +/- 0.0129
sequence
0.5012 +/- 0.0209
model_logprobs
0.6886 +/- 0.0185
rescaled_model_logprobs
0.6628 +/- 0.0175
task1
gpt-3.5-turbo-0125
express_distribution
0.2914 +/- 0.0125
sequence
0.3173 +/- 0.0120
model_logprobs
0.3395 +/- 0.0132
rescaled_model_logprobs
0.2895 +/- 0.0106
gpt-4
express_distribution
0.1812 +/- 0.0077
sequence
0.2388 +/- 0.0092
model_logprobs
0.5073 +/- 0.0144
rescaled_model_logprobs
0.2915 +/- 0.0107
anthropic_haiku
express_distribution
0.2221 +/- 0.0105
sequence
0.2842 +/- 0.0119
model_logprobs
nan +/- nan
rescaled_model_logprobs
nan +/- nan
anthropic_opus
express_distribution
0.2382 +/- 0.0122
sequence
0.2823 +/- 0.0124
model_logprobs
nan +/- nan
rescaled_model_logprobs
nan +/- nan
llama3-70b
express_distribution
0.2264 +/- 0.0098
sequence
0.3152 +/- 0.0129
model_logprobs
0.4601 +/- 0.0140
rescaled_model_logprobs
0.41

In [16]:
output_type_to_latex = {'express_distribution': "V", 'sequence': "Seq", 'model_logprobs': "Log-p", 'rescaled_model_logprobs': "TS-Log-p"}

models = ['gpt-3.5-turbo-0125', 'gpt-4', 'anthropic_haiku', 'anthropic_opus', 'llama3-70b']
output_types = ['express_distribution', 'sequence', 'model_logprobs', 'rescaled_model_logprobs']

for model in models: 
  print(model)
  for output_type in output_types: 
    print(output_type)
    
    opinionqa_data = df[(df.Model==model) & (df['Output Type']==output_type) & (df['Dataset']=='opinionqa') & ((df['Task Type']=='task1') | (df['Task Type']=='task3_easy_hard'))]['all_tvs']
    nytimes_data = df[(df.Model==model) & (df['Output Type']==output_type) & (df['Dataset']=='nytimes') & ((df['Task Type']=='task1') | (df['Task Type']=='task3_easy_hard'))]['all_tvs']
    mean, bs = compute_both(opinionqa_data, nytimes_data)

    # Adding a data entry
    new_entry = pd.DataFrame({
        'Model Name': ["{} ({})".format(model, output_type_to_latex[output_type])],
        'Dataset': ['both'],
        'Task Type': ['task1_task3'],
        'Alignment Mean': [mean],
        'Alignment Error': [bs]
    })

    # Adding the new entry to the dataframe
    final_results_df = pd.concat([final_results_df, new_entry], ignore_index=True)

    print('{:.4f} +/- {:.4f}'.format(mean, bs))

final_results_df.to_csv(final_results_path, index=False)

gpt-3.5-turbo-0125
express_distribution
0.2592 +/- 0.0049
sequence
0.2778 +/- 0.0053
model_logprobs
0.4621 +/- 0.0071
rescaled_model_logprobs
0.2703 +/- 0.0045
gpt-4
express_distribution
0.2041 +/- 0.0037
sequence
0.2374 +/- 0.0043
model_logprobs
0.5825 +/- 0.0059
rescaled_model_logprobs
0.3561 +/- 0.0053
anthropic_haiku
express_distribution
0.2347 +/- 0.0046
sequence
0.2872 +/- 0.0057
model_logprobs
nan +/- nan
rescaled_model_logprobs
nan +/- nan
anthropic_opus
express_distribution
0.2191 +/- 0.0048
sequence
0.3370 +/- 0.0055
model_logprobs
nan +/- nan
rescaled_model_logprobs
nan +/- nan
llama3-70b
express_distribution
0.2255 +/- 0.0043
sequence
0.3199 +/- 0.0055
model_logprobs
0.5144 +/- 0.0063
rescaled_model_logprobs
0.4598 +/- 0.0073


In [17]:
# UNIFORM AND DISCR ERROR

output_type_to_latex = {'express_distribution': "V", 'sequence': "Seq", 'model_logprobs': "Log-p", 'rescaled_model_logprobs': "TS-Log-p"}

task_types = ['ground_truth', 'uniform']
# output_types = ['express_distribution', 'sequence']
for task_type in task_types:
  for dataset in ['opinionqa', 'nytimes']:
    print(task_type)
    input_data = df[(df.Model=='simulated') & (df['Task Type']==task_type) & (df['Dataset']==dataset)]['all_tvs']
    mean, bs = compute_one(input_data)

    # Adding a data entry
    new_entry = pd.DataFrame({
        'Model Name': ['simulated'],
        'Dataset': [dataset],
        'Task Type': [task_type],
        'Alignment Mean': [mean],
        'Alignment Error': [bs]
    })

    # Adding the new entry to the dataframe
    final_results_df = pd.concat([final_results_df, new_entry], ignore_index=True)

    print('{:.4f} +/- {:.4f}'.format(mean, bs))


  # BOTH DATASETS
      
  opinionqa_data = df[(df.Model=='simulated') & (df['Task Type']==task_type) & (df['Dataset']=='opinionqa')]['all_tvs']
  nytimes_data = df[(df.Model=='simulated') & (df['Task Type']==task_type) & (df['Dataset']=='nytimes')]['all_tvs']
  mean, bs = compute_both(opinionqa_data, nytimes_data)

  # Adding a data entry
  new_entry = pd.DataFrame({
      'Model Name': ['simulated'],
      'Dataset': ['both'],
      'Task Type': [task_type],
      'Alignment Mean': [mean],
      'Alignment Error': [bs]
  })

  # Adding the new entry to the dataframe
  final_results_df = pd.concat([final_results_df, new_entry], ignore_index=True)

  print('{:.4f} +/- {:.4f}'.format(mean, bs))


ground_truth
0.1361 +/- 0.0119
ground_truth
0.1153 +/- 0.0007
0.1259 +/- 0.0057
uniform
0.3818 +/- 0.0085
uniform
0.2233 +/- 0.0062
0.3025 +/- 0.0049


#### Knowledge to Sim Gap

In [18]:
models = ['gpt-3.5-turbo-0125', 'gpt-4', 'anthropic_haiku', 'anthropic_opus', 'llama3-70b']
task_types = ['task1', 'task3_easy_hard']
output_types = ['']

for model in models: 
  print(model)
  all_KS = []
  for task_type in task_types:
    # OQA 
    dataset='opinionqa'
    for demographic_group in ['Democrat', 'Republican', 'Male',  'Female', 'Black', 'White']:
      seq=float(list(df[(df.Model==model) & (df['Output Type']=='sequence') & (df['Dataset']==dataset) & (df['Task Type']==task_type) & (df['Demographic/Avg']==demographic_group)]['TV'])[0])
      verb = float(list(df[(df.Model==model) & (df['Output Type']=='express_distribution') & (df['Dataset']==dataset) & (df['Task Type']==task_type) & (df['Demographic/Avg']==demographic_group)]['TV'])[0])
      all_KS.append((seq/verb)-1)

    dataset='nytimes'
    for demographic_group in ['Democrat', 'Republican', 'Male',  'Female']:
      seq=float(list(df[(df.Model==model) & (df['Output Type']=='sequence') & (df['Dataset']==dataset) & (df['Task Type']==task_type) & (df['Demographic/Avg']==demographic_group)]['TV'])[0])
      verb = float(list(df[(df.Model==model) & (df['Output Type']=='express_distribution') & (df['Dataset']==dataset) & (df['Task Type']==task_type) & (df['Demographic/Avg']==demographic_group)]['TV'])[0])
      all_KS.append((seq/verb)-1)
  
  mean, bs = compute_one(list(all_KS))
  # mean, bs = compute_both(all_KS_OQA, all_KS_NYT)
  # Adding a data entry
  new_entry = pd.DataFrame({
      'Model Name': [model],
      'Dataset': ['both'],
      'Task Type': ['knowledge_to_sim_gap'],
      'Alignment Mean': [mean],
      'Alignment Error': [bs]
  })

  # Adding the new entry to the dataframe
  final_results_df = pd.concat([final_results_df, new_entry], ignore_index=True)

  print('{:.4f} +/- {:.4f}'.format(mean, bs))


gpt-3.5-turbo-0125
0.0819 +/- 0.0462
gpt-4
0.1940 +/- 0.0700
anthropic_haiku
0.2471 +/- 0.0466
anthropic_opus
0.5355 +/- 0.1082
llama3-70b
0.4183 +/- 0.0413


### Humans

In [19]:
import json

#task_types = ['nosteer', 'persona', 'fewshot']
task_types = ['persona', 'fewshot']

with open('/Users/nicolemeister/Desktop/STANFORD/distributions/results/human_annotations/OQA_human_tv_data.json', 'r') as file:
    oqa_all_human_data = json.load(file)
    
with open('/Users/nicolemeister/Desktop/STANFORD/distributions/results/human_annotations/NYT_human_tv_data.json', 'r') as file:
    nyt_all_human_data = json.load(file)


oqa_human_data, nyt_human_data = [], []
for task_type in task_types:
    print(task_type)
    oqa_human_data.append(oqa_all_human_data[task_type]) # OQA
    nyt_human_data.append(nyt_all_human_data[task_type]) # NYT

    mean, bs = compute_both(oqa_all_human_data[task_type], nyt_all_human_data[task_type], human_data=True)
    print('{:.4f} +/- {:.4f}'.format(mean, bs))
    # Adding a data entry
    new_entry = pd.DataFrame({
        'Model Name': ['human'],
        'Dataset': ['both'],
        'Task Type': [task_type],
        'Alignment Mean': [mean],
        'Alignment Error': [bs]
    })

    # Adding the new entry to the dataframe
    final_results_df = pd.concat([final_results_df, new_entry], ignore_index=True)

combined_opinionqa_data = np.concatenate([oqa_all_human_data['persona'], oqa_all_human_data['fewshot']])
combined_nytimes_data = np.concatenate([nyt_all_human_data['persona'], nyt_all_human_data['fewshot']])


mean, bs = compute_both(combined_opinionqa_data, combined_nytimes_data, human_data=True)
print('AVGED: {:.4f} +/- {:.4f}'.format(mean, bs))
# Adding a data entry
new_entry = pd.DataFrame({
    'Model Name': ['simulated'],
    'Dataset': ['both'],
    'Task Type': ['task1_task3'],
    'Alignment Mean': [mean],
    'Alignment Error': [bs]
    })

  # Adding the new entry to the dataframe
final_results_df = pd.concat([final_results_df, new_entry], ignore_index=True)

final_results_df.to_csv(final_results_path, index=False)


persona
0.2597 +/- 0.0061
fewshot
0.2335 +/- 0.0063
AVGED: 0.2464 +/- 0.0045
